In [ ]:
# Standard library imports
from pathlib import Path
from typing import Optional, Tuple
import warnings

# Third-party imports
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import box
import folium
from folium import plugins
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

import ipyleaflet
from urllib.parse import urlencode
from pyproj import Transformer, CRS

# Ignore warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries imported successfully!")
print(f"\nLibrary Versions:")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Pandas: {pd.__version__}")
print(f"  Folium: {folium.__version__}")

In [ ]:
# Configuration Settings and Variables 

# Create data directory if it doesn't exist
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print(f"✓ Data directory ready: {DATA_DIR.absolute()}")
# API Endpoints
TOWN_BOUNDARIES_URL = (
    "https://services1.arcgis.com/BkFxaEFNwHqX3tAw/arcgis/rest/services/"
    "FS_VCGI_OPENDATA_Boundary_BNDHASH_poly_towns_SP_v1/FeatureServer/0/query"
    "?outFields=*&where=1%3D1&f=geojson"
)

GEOLOGY_MAPSERVICE_URL = (
    "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/"
    "OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/165"
)

GEOLOGY_QUERY_ENDPOINT = f"{GEOLOGY_MAPSERVICE_URL}/query"

BASEMAP_URL = (
    "https://basemaps.arcgis.com/arcgis/rest/services/"
    "World_Basemap_v2/VectorTileServer"
)

# Coordinate Reference Systems
# Note: GeoPandas accepts both "EPSG:####" and "####" formats
VT_STATE_PLANE = "EPSG:32145"  # Vermont State Plane NAD83 (meters)
WEB_MERCATOR = "EPSG:3857"      # Web Mercator (for tile services)
WGS84 = "EPSG:4326"             # WGS84 (latitude/longitude)

# File paths for cached data
TOWNS_CACHE = DATA_DIR / "towns.geojson"

print("✓ Configuration complete!")
print(f"\nData Sources:")
print(f"  Towns: VCGI OpenData Portal")
print(f"  Geology: VT Agency of Natural Resources")
print(f"\nCoordinate Systems:")
print(f"  Vermont State Plane: {VT_STATE_PLANE}")
print(f"  WGS84 (Web): {WGS84}")

In [ ]:
# Fetch Town Boundaries Function
def fetch_town_boundaries(use_cache: bool = True) -> gpd.GeoDataFrame:
    """
    Fetch Vermont town boundaries from VCGI OpenData portal.
    
    Parameters
    ----------
    use_cache : bool, default True
        If True, load from local cache if available. Otherwise, fetch from API.
    
    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing Vermont town boundaries in VT State Plane (EPSG:32145)
    
    Notes
    -----
    Data is cached to data/towns.geojson after first download.
    The source data is in WGS84 (lat/lon) and is transformed to VT State Plane (meters).
    """
    # Check if cached file exists and use_cache is True
    if use_cache and TOWNS_CACHE.exists():
        print(f"📁 Loading towns from cache: {TOWNS_CACHE}")
        gdf = gpd.read_file(TOWNS_CACHE)
        
        # IMPORTANT: GeoJSON is always in WGS84 (EPSG:4326)
        # We need to set the CRS and then transform to VT State Plane
        gdf = gdf.set_crs(WGS84, allow_override=True)
        gdf = gdf.to_crs(VT_STATE_PLANE)
        
        print(f"✓ Loaded {len(gdf)} towns from cache")
        print(f"✓ Transformed to {gdf.crs.name}")
        return gdf
    
    # Fetch from API
    print(f"🌐 Fetching town boundaries from VCGI...")
    print(f"   URL: {TOWN_BOUNDARIES_URL[:80]}...")
    
    try:
        # Make HTTP GET request
        response = requests.get(TOWN_BOUNDARIES_URL, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        
        # Parse GeoJSON response
        geojson_data = response.json()
        
        # Convert to GeoDataFrame
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])
        
        # Set coordinate reference system
        # GeoJSON spec requires WGS84 (EPSG:4326) coordinates
        gdf = gdf.set_crs(WGS84)
        
        print(f"✓ Fetched {len(gdf)} town boundaries in WGS84")
        
        # Transform to Vermont State Plane for easier measurement in meters
        # IMPORTANT: to_crs() returns a NEW GeoDataFrame, it doesn't modify in place!
        gdf = gdf.to_crs(VT_STATE_PLANE)
        
        print(f"✓ Transformed to {gdf.crs.name}")
        
        # Save to cache (will be saved in VT State Plane coordinates)
        print(f"💾 Saving to cache: {TOWNS_CACHE}")
        gdf.to_file(TOWNS_CACHE, driver='GeoJSON')
        print(f"✓ Cache saved successfully")
        
        return gdf
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing data: {e}")
        raise


In [ ]:
# Initiate town boundary fetch
towns_gdf = fetch_town_boundaries()

print("="*50)
print("TOWN BOUNDARIES LOADED SUCCESSFULLY")
print("="*50)

In [ ]:
# Read selected town data from dataframe
def on_town_selected(town_name: str) -> None:
    """
    Callback function when a town is selected from the dropdown.
    
    Parameters
    ----------
    town_name : str
        Name of the selected town
    """
    global selected_town_data, selected_town_name
    
    # Store the selected town name
    selected_town_name = town_name
    
    # Filter the GeoDataFrame to get the selected town
    selected_town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(selected_town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return
    
    # Get the first (and should be only) row
    town = selected_town_data.iloc[0]
    
    # Display town information
    print("=" * 70)
    print(f"📍 SELECTED TOWN: {town_name}")
    # print("=" * 70)
    
    # # Display key attributes
    # print(f"\nTown Information:")
    # for col in selected_town_data.columns:
    #     if col != 'geometry':
    #         value = town[col]
    #         print(f"  • {col}: {value}")
    
    # # Calculate and display area
    # # Area is calculated in square meters (since CRS is in meters)
    # area_sq_m = town.geometry.area
    # area_sq_km = area_sq_m / 1_000_000
    # area_acres = area_sq_m / 4046.86
    
    # print(f"\nGeographic Properties:")
    # print(f"  • Area: {area_sq_km:.2f} km² ({area_acres:.2f} acres)")
    
    # # Get bounding box
    # bounds = town.geometry.bounds
    # print(f"  • Bounding Box (Vermont State Plane, meters):")
    # print(f"      Min X: {bounds[0]:,.2f}")
    # print(f"      Min Y: {bounds[1]:,.2f}")
    # print(f"      Max X: {bounds[2]:,.2f}")
    # print(f"      Max Y: {bounds[3]:,.2f}")
    
    # # Calculate centroid
    # centroid = town.geometry.centroid
    # print(f"  • Centroid:")
    # print(f"      X: {centroid.x:,.2f}")
    # print(f"      Y: {centroid.y:,.2f}")
    
    # print("\n" + "=" * 70)
    print("✓ Town data loaded and ready for mapping")
    print("=" * 70)

In [ ]:
# Review Town Dataset - Name Field and content
town_name_field = "TOWNNAMEMC"

# Get sorted list of town names
town_names = sorted(towns_gdf[town_name_field].unique())

print(f"\n✓ Found {len(town_names)} Vermont towns")
print(f"\nTown name field: '{town_name_field}'")
print(f"\nSample towns (first 5):")
for name in town_names[:5]:
    print(f"  • {name}")
print(f"  ...")
print(f"  • {town_names[-1]}")

In [ ]:
# Display column names and types
print("Available Columns:")
print("=" * 60)
for col in towns_gdf.columns:
    dtype = towns_gdf[col].dtype
    if col != 'geometry':
        sample = towns_gdf[col].iloc[0] if len(towns_gdf) > 0 else None
        print(f"  {col:20s} ({dtype}) - Example: {sample}")
    else:
        print(f"  {col:20s} ({dtype})")

In [ ]:
# Display basic spatial information
print("Dataset Shape:")
print(f"  Rows (towns): {len(towns_gdf)}")
print(f"  Columns (attributes): {len(towns_gdf.columns)}")

print(f"\nCoordinate Reference System:")
print(f"  {towns_gdf.crs}")
print(f"  Name: {towns_gdf.crs.name}")

print(f"\nGeometry Type:")
print(f"  {towns_gdf.geometry.type.unique()}")

print(f"\nBounding Box (in meters, Vermont State Plane):")
bounds = towns_gdf.total_bounds
print(f"  Min X: {bounds[0]:,.2f}")
print(f"  Min Y: {bounds[1]:,.2f}")
print(f"  Max X: {bounds[2]:,.2f}")
print(f"  Max Y: {bounds[3]:,.2f}")

In [ ]:
# ════════════════CONTROL PANEL══════════════════════════════════════════════════
# INTERACTIVE CONTROL PANEL - PyDataVT 2025 Geospatial Demonstration
# ═══════════════════════════════════════════════════════════════════════════════

# Create a single output widget where all results will be displayed
output_panel = widgets.Output()

# Store global reference to geology data loaded by buttons
geology_gdf = None
geology_clipped_gdf = None

print("🎛️  VERMONT GEOLOGY ANALYSIS CONTROL PANEL")
print("=" * 80)
print("Select a town and use the buttons below to trigger different analyses.")
print("All outputs will appear in the panel below.")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Town Selector Dropdown
# ─────────────────────────────────────────────────────────────────────────────

town_dropdown = widgets.Dropdown(
    options=town_names,
    value=town_names[0],
    description='Select Town:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

def _on_town_change(change):
    """Update selected town when dropdown changes"""
    if change.get('name') == 'value' and change.get('new') is not None:
        with output_panel:
            output_panel.clear_output(wait=True)
            on_town_selected(change.get('new'))

town_dropdown.observe(_on_town_change, names='value')

# ─────────────────────────────────────────────────────────────────────────────
# Button: Fetch Geology Data
# ─────────────────────────────────────────────────────────────────────────────

def on_fetch_geology_click(b):
    """Fetch geology GeoJSON data for the selected town"""
    global geology_gdf, geology_colors

    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        print(f"📦 Fetching Geology Data for {selected_town_name}")
        print("=" * 70)

        # Fetch geology colors if not already loaded
        if 'geology_colors' not in globals() or not geology_colors:
            print("\n🎨 First, fetching official geology colors...")
            geology_colors = fetch_geology_colors()

        # Fetch geology data
        geology_gdf = fetch_geology_geojson(selected_town_name)

        if geology_gdf is not None:
            print(f"\n✅ SUCCESS!")
            print(f"   Geology data loaded: {len(geology_gdf)} features")
            print(f"   Total area: {geology_gdf.geometry.area.sum() / 1_000_000:.2f} km²")
            print(f"   Ready for analysis and visualization")

btn_fetch_geology = widgets.Button(
    description='📦 Fetch Geology Data',
    button_style='info',
    tooltip='Query geology data from ArcGIS REST API',
    layout=widgets.Layout(width='200px', height='40px')
)
btn_fetch_geology.on_click(on_fetch_geology_click)

# ─────────────────────────────────────────────────────────────────────────────
# Button: Show Static Map
# ─────────────────────────────────────────────────────────────────────────────

def on_static_map_click(b):
    """Generate static map image using MapServer export endpoint"""
    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        print(f"🗺️  Generating Static Map for {selected_town_name}")
        print("=" * 70)
        create_town_map(selected_town_name)

btn_static_map = widgets.Button(
    description='🗺️  Static Map',
    button_style='primary',
    tooltip='Generate static geology map image',
    layout=widgets.Layout(width='200px', height='40px')
)
btn_static_map.on_click(on_static_map_click)

# ─────────────────────────────────────────────────────────────────────────────
# Button: Show Interactive Map
# ─────────────────────────────────────────────────────────────────────────────

def on_interactive_map_click(b):
    """Create interactive ipyleaflet map with dynamic geology layer"""
    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        print(f"🌍 Creating Interactive Map for {selected_town_name}")
        print("=" * 70)
        interactive_map = create_interactive_map(selected_town_name)

        if interactive_map is not None:
            display(interactive_map)
            print("\n✅ Interactive map loaded!")
            print("   💡 Pan and zoom to explore - geology layer updates dynamically")

btn_interactive_map = widgets.Button(
    description='🌍 Interactive Map',
    button_style='success',
    tooltip='Create interactive map with pan/zoom',
    layout=widgets.Layout(width='200px', height='40px')
)
btn_interactive_map.on_click(on_interactive_map_click)

# ─────────────────────────────────────────────────────────────────────────────
# Button: Show Clipping & Visualization
# ─────────────────────────────────────────────────────────────────────────────

def on_clip_visualize_click(b):
    """Clip geology data to town boundary and visualize both on map"""
    global geology_gdf, geology_clipped_gdf, geology_colors

    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        # Ensure geology data is loaded
        if geology_gdf is None:
            print("📦 Geology data not loaded yet. Fetching now...")
            geology_gdf = fetch_geology_geojson(selected_town_name)
            if geology_gdf is None:
                print("❌ Failed to fetch geology data")
                return

        # Ensure colors are loaded
        if 'geology_colors' not in globals() or not geology_colors:
            geology_colors = fetch_geology_colors()

        print(f"✂️  Clipping & Visualizing Geology for {selected_town_name}")
        print("=" * 70)

        # Get town boundary
        town_boundary = selected_town_data.copy()

        print(f"\n🔄 Clipping {len(geology_gdf)} geology features to town boundary...")
        geology_clipped_gdf = gpd.clip(geology_gdf, town_boundary)
        print(f"✓ Clipped to {len(geology_clipped_gdf)} features")

        # Calculate areas
        area_before = geology_gdf.geometry.area.sum() / 1_000_000
        area_after = geology_clipped_gdf.geometry.area.sum() / 1_000_000
        town_area = town_boundary.geometry.area.sum() / 1_000_000

        print(f"\n📊 Area Comparison:")
        print(f"   Town boundary: {town_area:.2f} km²")
        print(f"   Full geology extent: {area_before:.2f} km²")
        print(f"   Clipped geology: {area_after:.2f} km²")
        print(f"   Reduction: {area_before - area_after:.2f} km² ({(1 - area_after/area_before)*100:.1f}%)")

        # Create map visualization
        print(f"\n🗺️  Creating interactive map with official colors...")

        # Convert to WGS84 for leaflet
        geology_wgs84 = geology_gdf.to_crs(WGS84)
        geology_clipped_wgs84 = geology_clipped_gdf.to_crs(WGS84)
        town_wgs84 = town_boundary.to_crs(WGS84)

        # Get center
        centroid = town_wgs84.geometry.centroid.iloc[0]
        center_lat, center_lon = centroid.y, centroid.x

        # Create map
        m = ipyleaflet.Map(
            center=(center_lat, center_lon),
            zoom=12,
            scroll_wheel_zoom=True,
            layout=widgets.Layout(width='100%', height='600px')
        )

        # Add basemap
        basemap = ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.OpenStreetMap.Mapnik)
        m.add_layer(basemap)

        # Style function for colored features
        def style_with_color(feature, bold=False):
            code = feature.get('properties', {}).get('CODE', None)
            if code and code in geology_colors:
                hex_color = rgba_to_hex(geology_colors[code])
            else:
                hex_color = '#cccccc' if not bold else '#999999'

            return {
                'color': hex_color,
                'fillColor': hex_color,
                'weight': 2 if bold else 1,
                'fillOpacity': 0.6
            }
        
        # Enhanced hover style for clipped layer
        def enhanced_hover_style(feature):
            """Enhanced hover style with brighter colors and thicker border"""
            code = feature.get('properties', {}).get('CODE', None)
            if code and code in geology_colors:
                hex_color = rgba_to_hex(geology_colors[code])
            else:
                hex_color = '#ffff00'  # Bright yellow fallback
            
            return {
                'color': '#ffffff',  # White border on hover
                'fillColor': hex_color,
                'weight': 5,  # Thick border
                'fillOpacity': 0.9  # More opaque
            }

        # Add layers
        geology_full_layer = ipyleaflet.GeoJSON(
            data=geology_wgs84.__geo_interface__,
            style_callback=lambda f: style_with_color(f, bold=False),
            hover_style={'weight': 3, 'fillOpacity': 0.7},
            name='Full Geology Extent'
        )
        m.add_layer(geology_full_layer)

        # Create HTML widget for displaying CODE on hover
        code_display = widgets.HTML(
            value='<div style="background: white; padding: 10px; border: 2px solid #333; border-radius: 5px; font-weight: bold;">Hover over clipped geology to see CODE</div>',
            layout=widgets.Layout(width='auto', height='auto')
        )
        
        # Function to update the CODE display on hover
        def on_hover_handler(**kwargs):
            """Update the HTML display when hovering over features"""
            if kwargs.get('type') == 'mouseover':
                feature = kwargs.get('feature', {})
                props = feature.get('properties', {})
                code = props.get('CODE', 'Unknown')
                lith = props.get('LITH', 'N/A')
                
                # Get color for this CODE
                if code in geology_colors:
                    hex_color = rgba_to_hex(geology_colors[code])
                    color_swatch = f'<span style="display: inline-block; width: 20px; height: 20px; background: {hex_color}; border: 1px solid black; margin-right: 5px;"></span>'
                else:
                    color_swatch = ''
                
                code_display.value = f'<div style="background: white; padding: 10px; border: 2px solid #333; border-radius: 5px; font-weight: bold; font-size: 14px;">{color_swatch}<strong>CODE:</strong> {code}<br><strong>LITH:</strong> {lith}</div>'
            elif kwargs.get('type') == 'mouseout':
                code_display.value = '<div style="background: white; padding: 10px; border: 2px solid #333; border-radius: 5px; font-weight: bold;">Hover over clipped geology to see CODE</div>'

        geology_clipped_layer = ipyleaflet.GeoJSON(
            data=geology_clipped_wgs84.__geo_interface__,
            style_callback=lambda f: style_with_color(f, bold=True),
            hover_style=enhanced_hover_style,
            name='Clipped Geology'
        )
        
        # Attach hover handler
        geology_clipped_layer.on_hover(on_hover_handler)
        geology_clipped_layer.on_msg(on_hover_handler)
        
        m.add_layer(geology_clipped_layer)

        town_layer = ipyleaflet.GeoJSON(
            data=town_wgs84.__geo_interface__,
            style={'color': '#cc0000', 'fillColor': 'transparent', 'weight': 3, 'fillOpacity': 0},
            name=f'{selected_town_name} Boundary'
        )
        m.add_layer(town_layer)

        # Add controls
        m.add_control(ipyleaflet.LayersControl(position='topright'))

        # Fit bounds
        bounds = town_wgs84.total_bounds
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

        # Display map and CODE display widget side by side
        map_and_info = widgets.HBox([m, code_display])
        display(map_and_info)

        print(f"\n✅ Visualization complete!")
        print(f"   🎨 Official MapServer colors applied")
        print(f"   🔴 Red outline: Town boundary")
        print(f"   💡 Hover over clipped geology features to see CODE")
        print(f"   Toggle layers using control in top-right")

btn_clip_viz = widgets.Button(
    description='✂️  Clip & Visualize',
    button_style='warning',
    tooltip='Clip geology to town boundary and show on map',
    layout=widgets.Layout(width='200px', height='40px')
)
btn_clip_viz.on_click(on_clip_visualize_click)

# ─────────────────────────────────────────────────────────────────────────────
# Button: Show Area Analysis Charts
# ─────────────────────────────────────────────────────────────────────────────

def on_area_analysis_click(b):
    """Generate area analysis charts for clipped geology data"""
    global geology_clipped_gdf, geology_colors

    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        # Ensure clipped data exists
        if geology_clipped_gdf is None:
            print("⚠️  No clipped geology data available.")
            print("   Please run 'Clip & Visualize' first!")
            return

        # Ensure colors are loaded
        if 'geology_colors' not in globals() or not geology_colors:
            geology_colors = fetch_geology_colors()

        print(f"📊 Area Analysis by Geology CODE for {selected_town_name}")
        print("=" * 70)

        # Calculate areas
        geology_clipped_gdf['area_km2'] = geology_clipped_gdf.geometry.area / 1_000_000
        area_by_code = geology_clipped_gdf.groupby('CODE')['area_km2'].sum().sort_values(ascending=False)

        # Get colors
        chart_colors = []
        for code in area_by_code.index:
            if code in geology_colors:
                rgba = rgba_to_mpl(geology_colors[code])
                chart_colors.append(rgba)
            else:
                chart_colors.append((0.8, 0.8, 0.8, 1.0))

        # Display table
        print(f"\n📋 Area by CODE:")
        print("-" * 70)
        print(f"{'CODE':<15} {'Color':<10} {'Area (km²)':<15} {'%':<10}")
        print("-" * 70)

        total_area = area_by_code.sum()
        for code, area in area_by_code.items():
            pct = (area / total_area) * 100
            hex_color = rgba_to_hex(geology_colors[code]) if code in geology_colors else '#cccccc'
            print(f"{code:<15} {hex_color:<10} {area:<15.2f} {pct:<10.1f}%")

        print("-" * 70)
        print(f"{'TOTAL':<15} {'':<10} {total_area:<15.2f} {'100.0%':<10}")

        # Create charts
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

        # Bar chart
        ax1.barh(range(len(area_by_code)), area_by_code.values, color=chart_colors, edgecolor='black', linewidth=0.5)
        ax1.set_yticks(range(len(area_by_code)))
        ax1.set_yticklabels(area_by_code.index)
        ax1.set_xlabel('Area (km²)', fontsize=12)
        ax1.set_ylabel('Geology CODE', fontsize=12)
        ax1.set_title(f'Geology Area by CODE - {selected_town_name}',
                      fontsize=14, fontweight='bold')
        ax1.grid(axis='x', alpha=0.3)
        ax1.invert_yaxis()

        for i, (code, area) in enumerate(area_by_code.items()):
            ax1.text(area, i, f' {area:.2f} km²', va='center', fontsize=9)

        # Pie chart
        wedges, texts, autotexts = ax2.pie(
            area_by_code.values,
            labels=area_by_code.index,
            autopct='%1.1f%%',
            startangle=90,
            colors=chart_colors,
            textprops={'fontsize': 10}
        )

        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
            autotext.set_fontsize(9)

        ax2.set_title(f'Geology Distribution - {selected_town_name}',
                      fontsize=14, fontweight='bold')

        plt.tight_layout()
        plt.show()

        print(f"\n✅ Analysis complete!")
        print(f"   {len(area_by_code)} unique geology codes")
        print(f"   Total area: {total_area:.2f} km²")

btn_area_analysis = widgets.Button(
    description='📊 Area Analysis',
    button_style='',
    tooltip='Generate area analysis charts',
    layout=widgets.Layout(width='200px', height='40px')
)
btn_area_analysis.on_click(on_area_analysis_click)

# ─────────────────────────────────────────────────────────────────────────────
# Button: Run All Analysis
# ─────────────────────────────────────────────────────────────────────────────

def on_run_all_click(b):
    """Run all analysis steps in sequence"""
    with output_panel:
        output_panel.clear_output(wait=True)

        if selected_town_name is None:
            print("⚠️  Please select a town first!")
            return

        print(f"🚀 RUNNING COMPLETE ANALYSIS FOR {selected_town_name}")
        print("=" * 80)

        # Step 1: Fetch geology data
        print("\n[1/4] 📦 Fetching Geology Data...")
        on_fetch_geology_click(None)

        # Step 2: Show static map
        print("\n\n[2/4] 🗺️  Generating Static Map...")
        print("-" * 70)
        create_town_map(selected_town_name)

        # Step 3: Clip and visualize
        print("\n\n[3/4] ✂️  Clipping & Visualizing...")
        print("-" * 70)
        on_clip_visualize_click(None)

        # Step 4: Area analysis
        print("\n\n[4/4] 📊 Area Analysis...")
        print("-" * 70)
        on_area_analysis_click(None)

        print("\n" + "=" * 80)
        print("✅ COMPLETE ANALYSIS FINISHED!")
        print("=" * 80)

btn_run_all = widgets.Button(
    description='🚀 RUN ALL ANALYSIS',
    button_style='danger',
    tooltip='Run all analysis steps in sequence',
    layout=widgets.Layout(width='220px', height='50px')
)
btn_run_all.on_click(on_run_all_click)

# ─────────────────────────────────────────────────────────────────────────────
# Layout and Display
# ─────────────────────────────────────────────────────────────────────────────

# Organize buttons in rows
button_row_1 = widgets.HBox([btn_fetch_geology, btn_static_map, btn_interactive_map])
button_row_2 = widgets.HBox([btn_clip_viz, btn_area_analysis, btn_run_all])

# Create the complete control panel
control_panel = widgets.VBox([
    town_dropdown,
    widgets.HTML("<br>"),
    widgets.HTML("<b>Analysis Tools:</b>"),
    button_row_1,
    button_row_2,
    widgets.HTML("<br><hr>"),
    widgets.HTML("<b>Output Panel:</b>"),
    output_panel
])

# Display the control panel
display(control_panel)

# Initialize with first town selected
_on_town_change({'name': 'value', 'new': town_names[0]})

In [ ]:
# Static Map Download Function
def create_town_map(town_name):
    """
    Create a static map image for the selected Vermont town using ArcGIS MapServer export.
    
    Parameters
    ----------
    town_name : str, optional
        Name of town to display. If None, uses globally selected town.
    
    Notes
    -----
    This uses the ArcGIS REST API 'export' endpoint to generate a static map image
    showing the geology layers for the town's bounding box.
    The town data is already in Vermont State Plane (EPSG:32145) coordinates,
    so we can use the bounds directly in meters.
    """
    # Use provided town name or fall back to global selection
    if town_name is None:
        if selected_town_name is None:
            print("⚠️  No town selected. Please select a town first.")
            return None
        town_name = selected_town_name
    
    # Get the town data
    town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return None
    
    # Get bounding box - town data is already in Vermont State Plane (EPSG:32145)
    # so bounds are already in meters
    bounds = town_data.total_bounds  # [minx, miny, maxx, maxy] in meters
    
    print(f"🗺️  Generating map for {town_name}...")
    print(f"   Town bounds (Vermont State Plane, meters):")
    print(f"      Min X (Easting):  {bounds[0]:,.2f}")
    print(f"      Min Y (Northing): {bounds[1]:,.2f}")
    print(f"      Max X (Easting):  {bounds[2]:,.2f}")
    print(f"      Max Y (Northing): {bounds[3]:,.2f}")
    
    # Add some padding (10% on each side) for better visualization
    width = bounds[2] - bounds[0]
    height = bounds[3] - bounds[1]
    padding_x = width * 0.1
    padding_y = height * 0.1
    
    # Create padded bounding box
    bbox_padded = [
        bounds[0] - padding_x,  # Min X (Easting)
        bounds[1] - padding_y,  # Min Y (Northing)
        bounds[2] + padding_x,  # Max X (Easting)
        bounds[3] + padding_y   # Max Y (Northing)
    ]
    
    # Format bbox as comma-separated string for the API
    bbox_str = ','.join(map(str, bbox_padded))
    
    print(f"   Padded bounds (with 10% buffer):")
    print(f"      {bbox_str}")
    
    # Construct the MapServer export URL
    base_url = "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/export"
    
    # Build query parameters
    # Key point: bboxSR and imageSR are both 32145 (Vermont State Plane)
    # This ensures the map service interprets our bbox correctly
    params = {
        'bbox': bbox_str,
        'bboxSR': '32145',  # Spatial Reference of the bbox (Vermont State Plane)
        'imageSR': '32145',  # Spatial Reference for the output image
        'size': '800,800',
        'dpi': '96',
        'format': 'png32',
        'transparent': 'true',
        
        'f': 'image'
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        
        # Display the image
        from IPython.display import Image as IPImage
        display(IPImage(response.content))
        
        print(f"\n✓ Map displayed for {town_name}")
        print(f"   Image size: 800x600 pixels")
        print(f"   Geology layer: Bedrock geology (Layer 165)")
        print(f"   Coordinate System: Vermont State Plane (EPSG:32145)")
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching map: {e}")
        raise

# # Create and display the map for the selected town
# if selected_town_name:
#     print(f"Creating map for: {selected_town_name}")
#     print("=" * 70)
#     create_town_map(selected_town_name)
# else:
#     print("⚠️  Please select a town from the dropdown above first!")

In [ ]:
# Interactive ipyleaflet Map

def create_interactive_map(town_name=None):
    """
    Create an interactive ipyleaflet map with ArcGIS MapServer geology layer.
    
    Parameters
    ----------
    town_name : str, optional
        Name of town to center map on. If None, uses globally selected town.
    
    Returns
    -------
    ipyleaflet.Map
        Interactive map with dynamic geology layer and town boundary
    
    Notes
    -----
    This demonstrates how to:
    - Create an ipyleaflet map
    - Add a dynamic ArcGIS MapServer layer using ImageOverlay
    - Reproject map bounds from WGS84 to VT State Plane for API calls
    - Add town boundary as GeoJSON layer
    - Handle map interactions (pan/zoom) to update the layer
    """
    # Use provided town name or fall back to global selection
    if town_name is None:
        if selected_town_name is None:
            print("⚠️  No town selected. Please select a town first.")
            return None
        town_name = selected_town_name
    
    # Get the town data
    town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()
    
    if len(town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return None
    
    # Convert town to WGS84 for ipyleaflet (which uses WGS84)
    town_wgs84 = town_data.to_crs(WGS84)
    
    # Get town center and bounds in WGS84
    centroid = town_wgs84.geometry.centroid.iloc[0]
    center_lat = centroid.y
    center_lon = centroid.x
    
    # Calculate bounds for fitting
    bounds_wgs = town_wgs84.total_bounds  # [minx, miny, maxx, maxy]
    
    MAP_PIXEL_WIDTH = 800
    MAP_PIXEL_HEIGHT = 600

    # Create the ipyleaflet map centered on the town
    m = ipyleaflet.Map(
        center=(center_lat, center_lon),
        zoom=12,
        scroll_wheel_zoom=True,
    )

    
    
    # Add a basemap layer
    basemap = ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.OpenStreetMap.Mapnik)
    m.add_layer(basemap)
    
    # Create placeholder ImageOverlay for the geology layer
    # Start with a transparent 1x1 pixel image
    image_overlay = ipyleaflet.ImageOverlay(
        url="data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkYAAAAAYAAjCB0C8AAAAASUVORK5CYII=",
        bounds=((bounds_wgs[1], bounds_wgs[0]), (bounds_wgs[3], bounds_wgs[2])),
        name="Geology Layer"
    )
    m.add_layer(image_overlay)
    
    # Add town boundary as GeoJSON layer
    # Convert to GeoJSON format that ipyleaflet expects
    town_geojson = town_wgs84.__geo_interface__
    
    geo_json_layer = ipyleaflet.GeoJSON(
        data=town_geojson,
        style={
            'color': '#cc0000',
            'fillColor': '#ffcccc',
            'weight': 3,
            'fillOpacity': 0.2
        },
        hover_style={
            'color': '#ff0000',
            'fillColor': '#ff6666',
            'weight': 4,
            'fillOpacity': 0.4
        },
        name=f'{town_name} Boundary'
    )
    m.add_layer(geo_json_layer)
    
    # Define the interaction handler to update the geology layer
    def handle_interaction(change):
        """
        Called on pan/zoom. Gets map bounds in WGS84, transforms to VT State Plane,
        and constructs the Esri export URL.
        
        Parameters
        ----------
        change : dict
            Dictionary containing the change information from the observer
        """
        # Get current map bounds from ipyleaflet (in WGS84)
        # Format: ((south, west), (north, east))
        map_bounds_wgs = m.bounds
        
        # Extract coordinates
        south, west = map_bounds_wgs[0]
        north, east = map_bounds_wgs[1]
        
        # Use pyproj transformer to convert corners to VT State Plane
        # Bottom-left corner
        xmin_proj, ymin_proj = transformer_wgs84_to_vtsp.transform(west, south)
        # Top-right corner  
        xmax_proj, ymax_proj = transformer_wgs84_to_vtsp.transform(east, north)
        
        # Construct the Esri export URL using the projected bounds
        bbox_str = f'{xmin_proj},{ymin_proj},{xmax_proj},{ymax_proj}'
        
        # Get current map size for proper resolution
        width, height = 800, 800  # Default size
        if hasattr(m, 'size') and m.size:
            width, height = m.size[0], m.size[1]
            print(f"Map size: {width}x{height}")
        
        params = {
            'bbox': bbox_str,
            'bboxSR': '32145',  # Vermont State Plane
            'imageSR': '32145',  # Request image in VT State Plane
            'size': f'{width},{height}',
            'dpi': '96',
            'format': 'png32',
            'transparent': 'true',
            'f': 'image'
        }
        
        # Construct full URL
        base_url = "https://anrmaps.vermont.gov/arcgis/rest/services/Open_Data/OPENDATA_ANR_GEOLOGIC_SP_NOCACHE_v2/MapServer/export"
        new_url = f"{base_url}?{urlencode(params)}"
        
        # Update the ImageOverlay
        # URL: Use the new dynamic URL
        # Bounds: Use the ORIGINAL WGS84 bounds to position the image on the map
        image_overlay.url = new_url
        image_overlay.bounds = map_bounds_wgs
    
    # Link the function to map interaction events
    # The observer will automatically trigger when the map bounds change
    # (including when fit_bounds is called below)
    m.observe(handle_interaction, names=['bounds'])
    
    # Add layer control
    layer_control = ipyleaflet.LayersControl(position='topright')
    m.add_control(layer_control)
    
    # Fit map to town bounds - this will trigger handle_interaction via the observer
    m.fit_bounds([[bounds_wgs[1], bounds_wgs[0]], [bounds_wgs[3], bounds_wgs[2]]])
    
    print(f"✓ Interactive map created for {town_name}")
    print(f"  Center: ({center_lat:.4f}, {center_lon:.4f})")
    print(f"  Layers: Basemap, Geology (dynamic), Town Boundary")
    print(f"  Pan and zoom to explore - geology layer updates dynamically!")
    
    map_layout = widgets.Layout(width='800px', height='800px')

    m.layout = map_layout

    return m

# # Create and display the interactive map with legend
# if selected_town_name:
#     print(f"Creating interactive map for: {selected_town_name}")
#     print("=" * 70)
#     interactive_map_display = create_interactive_map()
#     display(interactive_map_display)
# else:
#     print("⚠️  Please select a town from the dropdown above first!")

In [ ]:
# Query REST Endpoint for GeoJSON Export

def fetch_geology_geojson(town_name=None, use_cache=True):
    """
    Query the geology REST endpoint to get GeoJSON of all features that intersect
    the selected town's bounding box.

    Parameters
    ----------
    town_name : str, optional
        Name of town to query. If None, uses globally selected town.
    use_cache : bool, default True
        If True, cache results to avoid repeated API calls.

    Returns
    -------
    gpd.GeoDataFrame
        GeoDataFrame containing geology features in Vermont State Plane (EPSG:32145)

    Notes
    -----
    This uses the ArcGIS REST API 'query' endpoint which returns actual GeoJSON features
    (as opposed to the 'export' endpoint which returns a rendered image).

    The query uses:
    - geometry: Town bounding box in Vermont State Plane coordinates
    - geometryType: esriGeometryEnvelope (bounding box)
    - spatialRel: esriSpatialRelIntersects (find all features that intersect)
    - inSR: 32145 (Vermont State Plane)
    - outFields: * (all attributes)
    - returnGeometry: true (include geometry in response)
    - f: geojson (return as GeoJSON format)
    
    Cache files are saved in WGS84 (EPSG:4326) to comply with GeoJSON specification.
    """
    # Use provided town name or fall back to global selection
    if town_name is None:
        if selected_town_name is None:
            print("⚠️  No town selected. Please select a town first.")
            return None
        town_name = selected_town_name

    # Check cache first
    cache_file = DATA_DIR / f"geology_{town_name.replace(' ', '_')}.geojson"
    if use_cache and cache_file.exists():
        print(f"📁 Loading geology data from cache: {cache_file.name}")
        gdf = gpd.read_file(cache_file)
        # GeoJSON is always WGS84
        gdf = gdf.set_crs(WGS84, allow_override=True)
        # Transform to Vermont State Plane for analysis
        gdf = gdf.to_crs(VT_STATE_PLANE)
        print(f"✓ Loaded {len(gdf)} geology features from cache")
        print(f"✓ Transformed to {gdf.crs.name}")
        return gdf

    # Get the town data
    town_data = towns_gdf[towns_gdf[town_name_field] == town_name].copy()

    if len(town_data) == 0:
        print(f"⚠️  No data found for town: {town_name}")
        return None

    # Get bounding box in Vermont State Plane coordinates (meters)
    bounds = town_data.total_bounds  # [minx, miny, maxx, maxy]

    print(f"🌐 Querying geology data for {town_name}...")
    print(f"   Town bounds (Vermont State Plane, meters):")
    print(f"      {bounds[0]:,.2f}, {bounds[1]:,.2f}, {bounds[2]:,.2f}, {bounds[3]:,.2f}")

    # Build query parameters for ArcGIS REST API
    # Format the bounding box as comma-separated string
    geometry_bbox = f"{bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]}"

    params = {
        'geometry': geometry_bbox,
        'geometryType': 'esriGeometryEnvelope',  # Indicates we're using a bounding box
        'inSR': '32145',  # Input spatial reference (Vermont State Plane)
        'spatialRel': 'esriSpatialRelIntersects',  # Find features that intersect bbox
        'outFields': '*',  # Return all attribute fields
        'returnGeometry': 'true',  # Include geometry in response
        'f': 'geojson'  # Return as GeoJSON format
    }

    try:
        # Make the query request
        response = requests.get(GEOLOGY_QUERY_ENDPOINT, params=params, timeout=60)
        response.raise_for_status()

        # Parse GeoJSON response
        geojson_data = response.json()

        # Check if we got features
        if 'features' not in geojson_data or len(geojson_data['features']) == 0:
            print(f"⚠️  No geology features found for {town_name}")
            return None

        # Convert to GeoDataFrame
        gdf = gpd.GeoDataFrame.from_features(geojson_data['features'])

        # Set CRS (GeoJSON is always WGS84)
        gdf = gdf.set_crs(WGS84)

        print(f"✓ Fetched {len(gdf)} geology features in WGS84")

        # Save to cache BEFORE transforming (GeoJSON spec requires WGS84)
        print(f"💾 Saving to cache (WGS84): {cache_file.name}")
        gdf.to_file(cache_file, driver='GeoJSON')
        print(f"✓ Cache saved successfully")

        # Transform to Vermont State Plane for analysis
        gdf = gdf.to_crs(VT_STATE_PLANE)
        print(f"✓ Transformed to {gdf.crs.name}")

        # Display summary information
        print(f"\n📊 Geology Data Summary:")
        print(f"   Total features: {len(gdf)}")
        print(f"   Total area: {gdf.geometry.area.sum() / 1_000_000:.2f} km²")

        return gdf

    except requests.exceptions.RequestException as e:
        print(f"❌ Error querying data: {e}")
        raise
    except Exception as e:
        print(f"❌ Error processing data: {e}")
        raise


# Query and display geology data for the selected town
if selected_town_name:
    print(f"Querying geology data for: {selected_town_name}")
    print("=" * 70)
    geology_gdf = fetch_geology_geojson(selected_town_name)

    if geology_gdf is not None:
        print(f"\n✓ Geology GeoDataFrame created: 'geology_gdf'")
        print(f"   Shape: {geology_gdf.shape}")
        print(f"   CRS: {geology_gdf.crs.name}")
        
        # Display available columns
        print(f"\n📋 Available columns:")
        for col in geology_gdf.columns:
            if col != 'geometry':
                print(f"   • {col}")
else:
    print("⚠️  Please select a town from the dropdown above first!")

In [ ]:
# Fetch Geology Layer Renderer Information (Colors for Each CODE)

def fetch_geology_colors(use_cache=True):
    """
    Fetch the renderer information from the ArcGIS MapServer to get official colors
    for each geology CODE.
    
    Returns
    -------
    dict
        Dictionary mapping geology CODE to RGB color tuple (r, g, b, alpha)
    
    Notes
    -----
    The MapServer REST API provides a JSON response with renderer information at:
    drawingInfo.renderer.uniqueValueInfos[]
    
    Each entry contains:
    - value: The CODE field value
    - symbol.color: [r, g, b, alpha] array
    """
    # Check cache
    cache_file = DATA_DIR / "geology_colors.json"
    
    if use_cache and cache_file.exists():
        print(f"📁 Loading color mapping from cache: {cache_file.name}")
        import json
        with open(cache_file, 'r') as f:
            color_map = json.load(f)
        print(f"✓ Loaded colors for {len(color_map)} geology codes")
        return color_map
    
    # Fetch from API
    print("🎨 Fetching geology layer renderer information...")
    layer_info_url = f"{GEOLOGY_MAPSERVICE_URL}?f=json"
    
    try:
        response = requests.get(layer_info_url, timeout=30)
        response.raise_for_status()
        layer_info = response.json()
        
        # Extract the renderer information
        unique_value_infos = layer_info['drawingInfo']['renderer']['uniqueValueInfos']
        
        # Build color mapping: CODE -> (r, g, b, alpha)
        color_map = {}
        for info in unique_value_infos:
            code = info['value']
            color = info['symbol']['color']  # [r, g, b, alpha]
            color_map[code] = color
        
        print(f"✓ Fetched colors for {len(color_map)} geology codes")
        
        # Save to cache
        import json
        with open(cache_file, 'w') as f:
            json.dump(color_map, f, indent=2)
        print(f"💾 Saved color mapping to cache")
        
        return color_map
        
    except Exception as e:
        print(f"❌ Error fetching color information: {e}")
        return {}


def rgba_to_hex(rgba):
    """
    Convert RGBA color array [r, g, b, alpha] to hex string for web use.
    
    Parameters
    ----------
    rgba : list
        RGBA color values [r, g, b, alpha] where RGB are 0-255 and alpha is 0-255
    
    Returns
    -------
    str
        Hex color string like '#RRGGBB'
    """
    r, g, b = rgba[0], rgba[1], rgba[2]
    return f'#{r:02x}{g:02x}{b:02x}'


def rgba_to_mpl(rgba):
    """
    Convert RGBA color array [r, g, b, alpha] to matplotlib format.
    
    Parameters
    ----------
    rgba : list
        RGBA color values [r, g, b, alpha] where RGB are 0-255 and alpha is 0-255
    
    Returns
    -------
    tuple
        (r, g, b, alpha) tuple with values 0-1 for matplotlib
    """
    return (rgba[0]/255, rgba[1]/255, rgba[2]/255, rgba[3]/255)


# Fetch the color mapping
print("Fetching official geology colors from MapServer...")
print("=" * 70)
geology_colors = fetch_geology_colors()

# if geology_colors:
#     print(f"\n✓ Color mapping ready: 'geology_colors'")
#     print(f"\n📋 Sample colors:")
#     for code, rgba in list(geology_colors.items())[:5]:
#         hex_color = rgba_to_hex(rgba)
#         print(f"   {code}: {hex_color} {rgba}")
# else:
#     print("⚠️  Could not load color mapping")

In [ ]:
# Clip Geology Data to Town Boundary and Visualize on Map with Official Colors

if 'geology_gdf' in locals() and geology_gdf is not None and selected_town_name is not None:
    print("✂️  Clipping Geology Data to Town Boundary")
    print("=" * 70)
    
    # Get the selected town boundary
    town_boundary = selected_town_data.copy()
    
    # Ensure both datasets are in the same CRS (Vermont State Plane)
    print(f"Town CRS: {town_boundary.crs}")
    print(f"Geology CRS: {geology_gdf.crs}")
    
    # Perform the clip operation
    # This uses GeoPandas overlay to clip geology features to town boundary
    print(f"\n🔄 Clipping {len(geology_gdf)} geology features to town boundary...")
    
    geology_clipped = gpd.clip(geology_gdf, town_boundary)
    
    print(f"✓ Clipped to {len(geology_clipped)} features")
    
    # Calculate areas before and after clipping
    area_before = geology_gdf.geometry.area.sum() / 1_000_000
    area_after = geology_clipped.geometry.area.sum() / 1_000_000
    town_area = town_boundary.geometry.area.sum() / 1_000_000
    
    print(f"\n📊 Area Comparison:")
    print(f"   Town boundary area: {town_area:.2f} km²")
    print(f"   Full geology extent: {area_before:.2f} km²")
    print(f"   Clipped geology area: {area_after:.2f} km²")
    print(f"   Reduction: {area_before - area_after:.2f} km² ({(1 - area_after/area_before)*100:.1f}%)")
    
    # Create an interactive map with both datasets using official colors
    print(f"\n🗺️  Creating interactive map with official geology colors...")
    
    # Convert to WGS84 for leaflet display
    geology_wgs84 = geology_gdf.to_crs(WGS84)
    geology_clipped_wgs84 = geology_clipped.to_crs(WGS84)
    town_wgs84 = town_boundary.to_crs(WGS84)
    
    # Get town center for map
    centroid = town_wgs84.geometry.centroid.iloc[0]
    center_lat = centroid.y
    center_lon = centroid.x
    
    # Create map
    m = ipyleaflet.Map(
        center=(center_lat, center_lon),
        zoom=12,
        scroll_wheel_zoom=True,
        layout=widgets.Layout(width='100%', height='600px')
    )
    
    # Add basemap
    basemap = ipyleaflet.basemap_to_tiles(ipyleaflet.basemaps.OpenStreetMap.Mapnik)
    m.add_layer(basemap)
    
    # Function to create styled GeoJSON with official colors
    def style_feature_with_color(feature, use_clipped_style=False):
        """Style function for GeoJSON features using official geology colors"""
        code = feature.get('properties', {}).get('CODE', None)
        
        if code and code in geology_colors:
            hex_color = rgba_to_hex(geology_colors[code])
            # Use slightly darker edge color
            edge_color = hex_color
        else:
            # Fallback colors
            hex_color = '#cccccc' if not use_clipped_style else '#999999'
            edge_color = '#666666'
        
        return {
            'color': edge_color,
            'fillColor': hex_color,
            'weight': 2 if use_clipped_style else 1,
            'fillOpacity': 0.5
        }
    
    # Add full geology dataset with official colors
    geology_full_layer = ipyleaflet.GeoJSON(
        data=geology_wgs84.__geo_interface__,
        style_callback=lambda feature: style_feature_with_color(feature, use_clipped_style=False),
        hover_style={'weight': 3, 'fillOpacity': 0.7},
        name='Full Geology Extent'
    )
    m.add_layer(geology_full_layer)
    
    # Add clipped geology dataset with official colors (slightly bolder)
    geology_clipped_layer = ipyleaflet.GeoJSON(
        data=geology_clipped_wgs84.__geo_interface__,
        style_callback=lambda feature: style_feature_with_color(feature, use_clipped_style=True),
        hover_style={'weight': 4, 'fillOpacity': 0.8},
        name='Clipped Geology (Official Colors)'
    )
    m.add_layer(geology_clipped_layer)
    
    # Add town boundary (red outline)
    town_layer = ipyleaflet.GeoJSON(
        data=town_wgs84.__geo_interface__,
        style={
            'color': '#cc0000',
            'fillColor': 'transparent',
            'weight': 3,
            'fillOpacity': 0
        },
        name=f'{selected_town_name} Boundary'
    )
    m.add_layer(town_layer)
    
    # Add layer control
    layer_control = ipyleaflet.LayersControl(position='topright')
    m.add_control(layer_control)
    
    # Fit to town bounds
    bounds = town_wgs84.total_bounds
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    # Display the map
    print(f"✓ Map created with {len(m.layers)} layers")
    print(f"\n🎨 Layer Legend:")
    print(f"   Official geology colors from MapServer renderer")
    print(f"   Full geology extent (lighter) and clipped geology (bolder)")
    print(f"   🔴 Red outline: Town boundary")
    print(f"\n   Toggle layers on/off using the layer control in the top-right")
    
    # display(m)
    
    # Store clipped data globally for use in charts
    geology_clipped_gdf = geology_clipped
    
    print(f"\n✓ Clipped GeoDataFrame created: 'geology_clipped_gdf'")
    print(f"   Shape: {geology_clipped_gdf.shape}")
    print(f"   CRS: {geology_clipped_gdf.crs.name}")
    
else:
    print("⚠️  No geology data or town selection available.")
    print("   Please run the previous cells to fetch geology data first!")

In [ ]:
# # Analyze and Visualize Geology Area by CODE with Official Colors

# if 'geology_clipped_gdf' in locals() and geology_clipped_gdf is not None:
#     print("📊 Area Analysis by Geology CODE (Clipped to Town Boundary)")
#     print("=" * 70)
    
#     # Calculate area in square kilometers for each feature
#     # Note: geometry is in Vermont State Plane (meters), so area is in m²
#     geology_clipped_gdf['area_km2'] = geology_clipped_gdf.geometry.area / 1_000_000
    
#     # Group by CODE and sum the areas
#     area_by_code = geology_clipped_gdf.groupby('CODE')['area_km2'].sum().sort_values(ascending=False)
    
#     # Get official colors for each CODE in the chart
#     chart_colors = []
#     for code in area_by_code.index:
#         if code in geology_colors:
#             # Convert to matplotlib RGBA format
#             rgba = rgba_to_mpl(geology_colors[code])
#             chart_colors.append(rgba)
#         else:
#             # Fallback to gray if color not found
#             chart_colors.append((0.8, 0.8, 0.8, 1.0))
    
#     # Display table with color indicators
#     print(f"\n📋 Area by Geology CODE (within {selected_town_name}):")
#     print("-" * 70)
#     print(f"{'CODE':<15} {'Color':<10} {'Area (km²)':<15} {'Area (%)':<15}")
#     print("-" * 70)
    
#     total_area = area_by_code.sum()
#     for code, area in area_by_code.items():
#         pct = (area / total_area) * 100
#         if code in geology_colors:
#             hex_color = rgba_to_hex(geology_colors[code])
#         else:
#             hex_color = '#cccccc'
#         print(f"{code:<15} {hex_color:<10} {area:<15.2f} {pct:<15.1f}%")
    
#     print("-" * 70)
#     print(f"{'TOTAL':<15} {'':<10} {total_area:<15.2f} {'100.0%':<15}")
#     print()
    
#     # Create visualization with official colors
#     fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
#     # Bar chart with official colors
#     bars = ax1.barh(range(len(area_by_code)), area_by_code.values, color=chart_colors, edgecolor='black', linewidth=0.5)
#     ax1.set_yticks(range(len(area_by_code)))
#     ax1.set_yticklabels(area_by_code.index)
#     ax1.set_xlabel('Area (km²)', fontsize=12)
#     ax1.set_ylabel('Geology CODE', fontsize=12)
#     ax1.set_title(f'Geology Area by CODE - {selected_town_name}\n', 
#                   fontsize=14, fontweight='bold')
#     ax1.grid(axis='x', alpha=0.3)
#     ax1.invert_yaxis()  # Largest at top
    
#     # Add value labels on bars
#     for i, (code, area) in enumerate(area_by_code.items()):
#         ax1.text(area, i, f' {area:.2f} km²', va='center', fontsize=9)
    
#     # Pie chart with official colors
#     wedges, texts, autotexts = ax2.pie(
#         area_by_code.values, 
#         labels=area_by_code.index,
#         autopct='%1.1f%%',
#         startangle=90,
#         colors=chart_colors,
#         textprops={'fontsize': 10}
#     )
    
#     # Make percentage text bold and ensure visibility
#     for autotext in autotexts:
#         autotext.set_color('white')
#         autotext.set_fontweight('bold')
#         autotext.set_fontsize(9)
    
#     ax2.set_title(f'Geology Distribution - {selected_town_name}\n', 
#                   fontsize=14, fontweight='bold')
    
#     plt.tight_layout()
#     plt.show()
    
#     print(f"\n✓ Charts generated with official geology colors")
#     print(f"   {len(area_by_code)} unique geology codes")
#     print(f"   Total clipped area: {total_area:.2f} km²")
#     print(f"   Number of features: {len(geology_clipped_gdf)}")
#     print(f"\n   Colors match the MapServer legend and map display")
    
# else:
#     print("⚠️  No clipped geology data available.")
#     print("   Please run the clipping cell above first!")